In [43]:
import pandas as pd
import numpy as np

file_path = "../data/raw/CanAI Cafe 2023 Sales Information.csv"

df = pd.read_csv(file_path)

df.head()

,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date,Province
0,TXN_9687814,coffee,2,3.5,7.0,Digital Wallet,In-store,2023-09-28,British Columbia
1,TXN_7002925,Refresher,2,5.0,10.0,Cash,NaN,2023-05-02,NL
2,TXN_7668262,Donut,1,2.0,2.0,Digital Wallet,In-store,2023-11-27,British Columbia
3,TXN_8322624,Refresher,3,5.0,15.0,ERR_PM_102,In-store,NaN,Saskatchewan
4,TXN_9390285,Coffee,1,3.5,3.5,ERR_PM_102,Takeaway,2023-05-11,newfoundland and labrador


In [44]:
df_clean = df.copy()

# Convert blank entries like "", " ", "   " into NaN
df_clean = df_clean.replace(r"^\s*$", np.nan, regex=True)

In [22]:
print("Province values before cleaning:")
print(df_clean["Province"].value_counts(dropna=False).to_string())

Province values before cleaning:
Province
British Columbia             2780
Newfoundland                 1992
Saskatchewan                 1644
Manitoba                     1612
NaN                           417
british columbia              127
New Foundland                 124
BritishColumbia               124
NFLD                          122
newfoundland and labrador     111
Manitba                       110
British Columba               107
B.C.                          106
manitoba                       82
Saskatchewn                    81
saskatchewan                   78
Manitobaa                      73
Sasktchewan                    71
Sask.                          71
SK                             27
BC                             25
MB                             22
NL                             20
BRITISH COLUMBIA                8
British Columbi                 6
Newfoundland                    5
newfoundland                    5
NEWFOUNDLAND                    5
Ont.  

In [23]:

province_mapping = {
    # British Columbia
    "british columbia": "British Columbia",
    "britishcolumbia": "British Columbia",
    "british columba": "British Columbia",
    "british columbi": "British Columbia",
    "bc": "British Columbia",
    "b c": "British Columbia",

    # Newfoundland and Labrador
    "newfoundland": "Newfoundland and Labrador",
    "new foundland": "Newfoundland and Labrador",
    "newfoundland and labrador": "Newfoundland and Labrador",
    "nfld": "Newfoundland and Labrador",
    "nl": "Newfoundland and Labrador",
    "newfoundlan": "Newfoundland and Labrador",

    # Manitoba
    "manitoba": "Manitoba",
    "manitba": "Manitoba",
    "manitobaa": "Manitoba",
    "manitob": "Manitoba",
    "mb": "Manitoba",

    # Saskatchewan
    "saskatchewan": "Saskatchewan",
    "saskatchewn": "Saskatchewan",
    "sasktchewan": "Saskatchewan",
    "saskatchewa": "Saskatchewan",
    "sask": "Saskatchewan",
    "sk": "Saskatchewan",

    # Ontario
    "ontario": "Ontario",
    "ontaroi": "Ontario",
    "ontairo": "Ontario",
    "ont": "Ontario",
    "on": "Ontario"
}

def clean_province(value):
    if pd.isna(value):
        return np.nan

    value = str(value).strip().lower()
    value = value.replace(".", "")
    value = " ".join(value.split())

    return province_mapping.get(value, value.title())

df_clean["Province_cleaned"] = df_clean["Province"].apply(clean_province)

df_clean["Province_cleaned"].value_counts(dropna=False)

Province_cleaned
British Columbia             3288
Newfoundland and Labrador    2387
Saskatchewan                 1979
Manitoba                     1909
NaN                           417
Ontario                        20
Name: count, dtype: int64

In [24]:
df["Item"].value_counts(dropna=False).head(100)

Item
Coffee       2248
Tea          1257
Sandwich     1060
Cookie        960
Donut         936
Refresher     810
Salad         748
Juice         523
Sandwhich     119
cofee         112
coffe         112
sandwich      111
coffee        102
Cofee         100
C0ffee         97
NaN            87
Tee            78
tee            78
Tea            77
donut          67
Doughnut       64
TEA            63
Donutt         56
Juic           44
juice          43
Juicee         35
               13
Name: count, dtype: int64

In [25]:
import re

item_mapping = {
    # Coffee
    "coffee": "Coffee",
    "cofee": "Coffee",
    "coffe": "Coffee",
    "c0ffee": "Coffee",

    # Tea
    "tea": "Tea",
    "tee": "Tea",

    # Sandwich
    "sandwich": "Sandwich",
    "sandwhich": "Sandwich",

    # Cookie
    "cookie": "Cookie",

    # Donut
    "donut": "Donut",
    "doughnut": "Donut",
    "donutt": "Donut",

    # Refresher
    "refresher": "Refresher",

    # Salad
    "salad": "Salad",

    # Juice
    "juice": "Juice",
    "juic": "Juice",
    "juicee": "Juice",
}

def clean_item(value):
    if pd.isna(value):
        return np.nan

    value = str(value).strip().lower()

    # Treat empty strings as missing
    if value == "":
        return np.nan

    # Normalize spacing and punctuation
    value = value.replace(".", "")
    value = value.replace("-", " ")
    value = value.replace("_", " ")
    value = re.sub(r"\s+", " ", value)

    return item_mapping.get(value, value.title())

df_clean["Item_cleaned"] = df_clean["Item"].apply(clean_item)

df_clean["Item_cleaned"].value_counts(dropna=False)

Item_cleaned
Coffee       2771
Tea          1553
Sandwich     1290
Donut        1123
Cookie        960
Refresher     810
Salad         748
Juice         645
NaN           100
Name: count, dtype: int64

In [27]:
df_clean["Quantity"] = pd.to_numeric(df_clean["Quantity"], errors="coerce").astype(float)

# Create flag before imputing
df_clean["Quantity_imputed_flag"] = df_clean["Quantity"].isna().astype(int)

# Rows where Quantity is missing but Price Per Unit and Total Spent are present
quantity_impute_mask = (
    df_clean["Quantity"].isna()
    & df_clean["Price Per Unit"].notna()
    & df_clean["Total Spent"].notna()
    & (df_clean["Price Per Unit"] != 0)
)

# Impute Quantity
df_clean.loc[quantity_impute_mask, "Quantity"] = (
    df_clean.loc[quantity_impute_mask, "Total Spent"]
    / df_clean.loc[quantity_impute_mask, "Price Per Unit"]
)

In [29]:
df_clean.loc[
    quantity_impute_mask,
    ["Price Per Unit", "Total Spent", "Quantity", "Quantity_imputed_flag"]
].head(20)

,Price Per Unit,Total Spent,Quantity,Quantity_imputed_flag
38,8.0,8.0,1.0,1
91,3.5,3.5,1.0,1
290,4.5,9.0,2.0,1
314,2.0,2.0,1.0,1
347,3.5,7.0,2.0,1
390,3.0,3.0,1.0,1
581,2.0,4.0,2.0,1
598,2.0,2.0,1.0,1
742,8.0,24.0,3.0,1
960,3.5,14.0,4.0,1


In [30]:
df_clean

,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date,Province,Province_cleaned,Item_cleaned,Quantity_imputed_flag
0,TXN_9687814,coffee,2.0,3.5,7.0,Digital Wallet,In-store,2023-09-28,British Columbia,British Columbia,Coffee,0
1,TXN_7002925,Refresher,2.0,5.0,10.0,Cash,NaN,2023-05-02,NL,Newfoundland and Labrador,Refresher,0
2,TXN_7668262,Donut,1.0,2.0,2.0,Digital Wallet,In-store,2023-11-27,British Columbia,British Columbia,Donut,0
3,TXN_8322624,Refresher,3.0,5.0,15.0,ERR_PM_102,In-store,NaN,Saskatchewan,Saskatchewan,Refresher,0
4,TXN_9390285,Coffee,1.0,3.5,3.5,ERR_PM_102,Takeaway,2023-05-11,newfoundland and labrador,Newfoundland and Labrador,Coffee,0
...,...,...,...,...,...,...,...,...,...,...,...,...
9995,TXN_1600387,Coffee,1.0,3.5,3.5,Cash,In-store,2023-03-02,newfoundland and labrador,Newfoundland and Labrador,Coffee,0
9996,TXN_9820954,Doughnut,4.0,2.0,8.0,Credit Card,Takeaway,2023-04-16,british columbia,British Columbia,Donut,0
9997,TXN_1197795,Tea,1.0,3.0,3.0,Credit Card,In-store,2023-02-07,British Columbia,British Columbia,Tea,0
9998,TXN_3269811,Juice,1.0,4.5,4.5,Digital Wallet,Takeaway,2023-06-16,Saskatchewan,Saskatchewan,Juice,0


In [31]:
transaction_id_col = "Transaction ID"

duplicate_id_counts = (
    df_clean[transaction_id_col]
    .value_counts(dropna=False)
    .reset_index()
)

duplicate_id_counts.columns = [transaction_id_col, "count"]

duplicate_id_counts[duplicate_id_counts["count"] > 1].head(20)

,Transaction ID,count
0,TXN_3000725,2
1,TXN_3341273,2
2,TXN_6066756,2
3,TXN_2817173,2
4,TXN_3134203,2
5,TXN_2676077,2
6,TXN_8355751,2
7,TXN_7165430,2
8,TXN_5333530,2
9,TXN_9655896,2


In [32]:
duplicate_transaction_ids = duplicate_id_counts[
    duplicate_id_counts["count"] > 1
][transaction_id_col]

duplicate_rows = df_clean[df_clean[transaction_id_col].isin(duplicate_transaction_ids)]

print("Number of duplicate Transaction IDs:", duplicate_transaction_ids.shape[0])
print("Number of rows involved in duplicate Transaction IDs:", duplicate_rows.shape[0])
print("Percentage of dataset involved:", round((duplicate_rows.shape[0] / df_clean.shape[0]) * 100, 2), "%")

Number of duplicate Transaction IDs: 44
Number of rows involved in duplicate Transaction IDs: 88
Percentage of dataset involved: 0.88 %


In [33]:
duplicate_rows_sorted = duplicate_rows.sort_values(transaction_id_col)

duplicate_rows_sorted.head(50)

,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date,Province,Province_cleaned,Item_cleaned,Quantity_imputed_flag
7108,TXN_1035070,Tea,2.0,3.0,6.0,Digital Wallet,Takeaway,2023-04-28,Manitoba,Manitoba,Tea,0
7107,TXN_1035070,Refresher,1.0,5.0,5.0,Credit Card,Takeaway,2023-03-28,Manitoba,Manitoba,Refresher,0
9201,TXN_1485560,Coffee,4.0,3.5,14.0,Cash,Takeaway,2023-07-17,British Columbia,British Columbia,Coffee,0
9200,TXN_1485560,Cookie,1.0,2.5,2.5,NaN,In-store,2023-02-07,manitoba,Manitoba,Cookie,0
6005,TXN_1834396,Refresher,1.0,5.0,5.0,Credit Card,In-store,2023-09-03,B.C.,British Columbia,Refresher,0
8949,TXN_1834396,Salad,2.0,9.0,18.0,NaN,In-store,2023-01-05,MANITOBA,Manitoba,Salad,0
9541,TXN_1954517,Coffee,1.0,3.5,3.5,Digital Wallet,Takeaway,2023-03-03,Newfoundland,Newfoundland and Labrador,Coffee,0
9542,TXN_1954517,Cookie,4.0,2.5,10.0,Digital Wallet,In-store,2023-05-16,BritishColumbia,British Columbia,Cookie,0
4752,TXN_2009944,Donut,2.0,2.0,4.0,NaN,Takeaway,2023-10-18,Manitba,Manitoba,Donut,0
4753,TXN_2009944,Tea,4.0,3.0,12.0,Digital Wallet,In-store,2023-05-24,Newfoundland,Newfoundland and Labrador,Tea,0


In [34]:
exact_duplicate_rows = df_clean[df_clean.duplicated(keep=False)]

print("Exact duplicate rows:", exact_duplicate_rows.shape[0])

exact_duplicate_rows.sort_values(transaction_id_col).head(50)

Exact duplicate rows: 0


,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date,Province,Province_cleaned,Item_cleaned,Quantity_imputed_flag


In [35]:
columns_to_check = [
    col for col in df_clean.columns
    if col != transaction_id_col
]

duplicate_group_summary = (
    duplicate_rows
    .groupby(transaction_id_col)[columns_to_check]
    .nunique(dropna=False)
    .reset_index()
)

duplicate_group_summary.head()

,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date,Province,Province_cleaned,Item_cleaned,Quantity_imputed_flag
0,TXN_1035070,2,2,2,2,2,1,2,1,1,2,1
1,TXN_1485560,2,2,2,2,2,2,2,2,2,2,1
2,TXN_1834396,2,2,2,2,2,1,2,2,2,2,1
3,TXN_1954517,2,2,2,2,1,2,2,2,2,2,1
4,TXN_2009944,2,2,2,2,2,2,2,2,2,2,1


In [36]:
conflicting_duplicate_ids = duplicate_group_summary[
    duplicate_group_summary[columns_to_check].gt(1).any(axis=1)
]

print("Duplicate Transaction IDs with conflicting values:", conflicting_duplicate_ids.shape[0])

conflicting_duplicate_ids.head(20)

Duplicate Transaction IDs with conflicting values: 44


,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date,Province,Province_cleaned,Item_cleaned,Quantity_imputed_flag
0,TXN_1035070,2,2,2,2,2,1,2,1,1,2,1
1,TXN_1485560,2,2,2,2,2,2,2,2,2,2,1
2,TXN_1834396,2,2,2,2,2,1,2,2,2,2,1
3,TXN_1954517,2,2,2,2,1,2,2,2,2,2,1
4,TXN_2009944,2,2,2,2,2,2,2,2,2,2,1
5,TXN_2138721,2,2,2,2,2,1,2,2,2,2,1
6,TXN_2459672,2,2,2,2,2,2,2,2,2,2,1
7,TXN_2543928,2,2,2,2,2,2,2,2,2,2,1
8,TXN_2550310,2,1,2,2,1,2,2,1,1,2,1
9,TXN_2676077,2,2,2,2,2,2,2,2,2,2,1


In [37]:
conflict_details = []

for _, row in conflicting_duplicate_ids.iterrows():
    transaction_id = row[transaction_id_col]

    conflicting_cols = [
        col for col in columns_to_check
        if row[col] > 1
    ]

    conflict_details.append({
        transaction_id_col: transaction_id,
        "conflicting_columns": conflicting_cols,
        "number_of_conflicting_columns": len(conflicting_cols)
    })

conflict_details_df = pd.DataFrame(conflict_details)

conflict_details_df.sort_values(
    "number_of_conflicting_columns",
    ascending=False
).head(20)

,Transaction ID,conflicting_columns,number_of_conflicting_columns
1,TXN_1485560,"[Item, Quantity, Price Per Unit, Total Spent, ...",10
7,TXN_2543928,"[Item, Quantity, Price Per Unit, Total Spent, ...",10
6,TXN_2459672,"[Item, Quantity, Price Per Unit, Total Spent, ...",10
4,TXN_2009944,"[Item, Quantity, Price Per Unit, Total Spent, ...",10
9,TXN_2676077,"[Item, Quantity, Price Per Unit, Total Spent, ...",10
32,TXN_6839885,"[Item, Quantity, Price Per Unit, Total Spent, ...",10
20,TXN_5162336,"[Item, Quantity, Price Per Unit, Total Spent, ...",10
27,TXN_6137624,"[Item, Quantity, Price Per Unit, Total Spent, ...",10
18,TXN_4725338,"[Item, Quantity, Price Per Unit, Total Spent, ...",10
15,TXN_3341273,"[Item, Quantity, Price Per Unit, Total Spent, ...",10


In [42]:
df_clean.to_csv("../data/interim/CanAI_Cafe_2023_Sales_Information_Step1.csv", index=False)